RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [16]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source inoformation to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: artificial_intelligence.pdf
Loaded 44 pages

Processing: machine_learning.pdf
Loaded 113 pages

Total documents loaded: 157


In [17]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf_files\\artificial_intelligence.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': 'artificial_intelligence.pdf', 'file_type': 'pdf'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 1\nINTRODUCTION \nTO ARTIFICIAL \nINTELLIGENCE (AI) \nTECHNOLOGY\nGUIDE FOR TRAVEL & TOURISM LEADERS\nJanuary 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf_files\\artificial_intelligence.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2', 'source_file': 'artificial_intelligence.pdf', 'file_type': 'pdf'}, page_content='INTRODUCTION

In [18]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [19]:
chunks = split_documents(all_pdf_documents)
chunks

Split 157 documents into 381 chunks

Example chunk:
Content: INTRODUCTION TO AI
World Travel & Tourism Council
< Contents  | 1
INTRODUCTION 
TO ARTIFICIAL 
INTELLIGENCE (AI) 
TECHNOLOGY
GUIDE FOR TRAVEL & TOURISM LEADERS
January 2024...
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf_files\\artificial_intelligence.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': 'artificial_intelligence.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf_files\\artificial_intelligence.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': 'artificial_intelligence.pdf', 'file_type': 'pdf'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 1\nINTRODUCTION \nTO ARTIFICIAL \nINTELLIGENCE (AI) \nTECHNOLOGY\nGUIDE FOR TRAVEL & TOURISM LEADERS\nJanuary 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf_files\\artificial_intelligence.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2', 'source_file': 'artificial_intelligence.pdf', 'file_type': 'pdf'}, page_content='INTRODUCTION

Embedding and VectorStore DB

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import list, dict, any, tuple
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: No module named 'sentence_transformers'